In [41]:
#!/usr/bin/env python3
import os
import pandas as pd

def reverse_readline(fh, buf_size=8192):
    """
    A generator that returns the lines of a file in reverse order.
    Reads file by blocks from the end. (Works for text files opened in binary mode.)
    """
    segment = None
    offset = 0
    fh.seek(0, os.SEEK_END)
    position = fh.tell() # position where the file was read
    file_size = fh.tell()
    while offset < file_size:
        # Incrementally decrease the position of read
        offset = min(file_size, offset + buf_size)
        position = file_size - offset
        fh.seek(position)

        # Read a buffer size of data
        buffer = fh.read(min(buf_size, offset))

        # split the buffer by newline
        lines = buffer.split(b'\n')

        # The first segment of the current buffer is likely a partial line, so add it to the previous segment.
        if segment is not None:
            if buffer[-1] != ord(b'\n'):
                # If the last char isn't a newline, then the last element is partial.
                lines[-1] += segment
            else:
                lines.append(segment)
        segment = lines[0]

        for line in reversed(lines[1:]):
            line = line.decode('utf-8', errors='replace')
            yield line.strip(), position
    # yield the last remaining segment
    if segment is not None:
        segment = segment.decode('utf-8', errors='replace')
        yield segment.strip(), position

def forward_read_lines(header, date_time, log_file_path, start_offset, num_lines=50):
    """
    Open the file in forward (normal) mode, seek to start_offset, and read num_lines.
    Returns a list of strings (lines).
    """
    lines = []
    with open(log_file_path, "r", encoding="utf-8", errors="replace") as f:
        f.seek(start_offset - 81920) # make sure enough lines are read

        marker_found = False
        count = 0
        while (count < num_lines):
            line = f.readline()

            if header in line and date_time in line:
                marker_found = True
                continue

            if marker_found:
                lines.append(line.rstrip("\n"))
                count = count + 1
    return lines


def parse_table_line(line, header):
    """
    Given a table line starting with <i> <j> <k> ..., parse it into tokens.
    """
    tokens = line.split()

    if header == "Post-reaction cec cation":
        index = ['c','j','icat','cec_cation_vr', 'meq cec_cation_vr', 
                 '-cec_cation_flux_vr * dt', '-cec_cation_flux2_vr * dt', 
                 'background_cec_vr * dt']
    elif header == "Post-reaction cation":
        index = ['c','j','icat','cation_vr','mol cation_vr',
                 'background_flux_vr * dt', 'primary_cation_flux_vr * dt', 
                 'cec_cation_flux_vr * dt', 'cec_cation_flux2_vr * dt', 
                 '-secondary_cation_flux_vr * dt', 
                 '-cation_uptake_vr * dt', 'cation_infl_vr * dt', 
                 '-cation_leached_vr * dt', 'cation_runoff_vr * dt']

    tokens = pd.Series(tokens, index = index).astype(float)

    # You may want to convert tokens to appropriate types, e.g. int or float.
    # For now, we simply return the tokens.
    return tokens


if __name__ == '__main__':
    # Change this to your error log file path
    log_file_path = "/gpfs/wolf2/cades/cli185/proj-shared/ywo/E3SM/output/20250407_HBR_ICB20TRCNPRDCTCBC_6year_rmethod1erw/run/fort.100"

    # Set the problematic grid cell and time step
    latitude = '43.955669499999999' # f'{44.75:.15f}'
    longitude = '-71.728824999999972' # f'{360 - 290.75:.15f}'
    date_time = '1938-12-16_15:00:00'

    # Open the file in binary mode for reverse reading for actual info. 
    collected_lines = {"Post-reaction cec cation": [],
                        "Post-reaction cation": []}  # Will collect lines from diagnostics upward to the key marker
    n_soil_layers = 7 # HBR = 7, elsewhere = 10
    with open(log_file_path, "rb") as fh:
        # Read lines in reverse
        for line, position in reverse_readline(fh):
            filt = latitude in line and longitude in line and date_time in line
            if filt:
                if "Post-reaction cec cation" in line:
                    collected_lines["Post-reaction cec cation"] = forward_read_lines("Post-reaction cec cation", date_time, log_file_path, position, n_soil_layers * 5)
                    continue
                if "Post-reaction cation" in line:
                    collected_lines["Post-reaction cation"] = forward_read_lines("Post-reaction cation", date_time, log_file_path, position, n_soil_layers * 5)
                    break

    # Reverse collected_lines so that they are in original order (from "Post-reaction cec cation" down to diagnostics)
    for key in collected_lines.keys():
        table_lines = collected_lines[key]

        table_lines.reverse()

        for i, line in enumerate(table_lines):
            stripped = line.lstrip()
            if stripped and stripped[0].isdigit():
                table_lines[i] = parse_table_line(line, key)
            else:
                raise Exception("Table line not found after 'Post-reaction cec cation' marker.")

        table_lines = pd.DataFrame(table_lines)
        table_lines['c'] = table_lines['c'].astype(int)
        table_lines['j'] = table_lines['j'].astype(int)
        table_lines['icat'] = table_lines['icat'].astype(int)
        table_lines = table_lines.set_index(['c','j','icat']).sort_index()

        collected_lines[key] = table_lines


In [42]:
collected_lines['Post-reaction cation']

cation_vr  mol cation_vr  background_flux_vr * dt  \
c j icat                                                      
1 1 1           NaN            NaN             2.495804e-05   
    2           NaN            NaN             5.697402e-06   
    3           NaN            NaN             0.000000e+00   
    4           NaN            NaN             6.589666e-06   
    5           NaN            NaN             1.317414e-05   
  2 1           NaN            NaN             1.105492e-05   
    2           NaN            NaN             2.336130e-06   
    3           NaN            NaN             0.000000e+00   
    4           NaN            NaN             3.923492e-06   
    5           NaN            NaN             1.325275e-05   
  3 1           NaN            NaN             7.688976e-06   
    2           NaN            NaN             1.503027e-06   
    3           NaN            NaN             0.000000e+00   
    4           NaN            NaN             1.966740e-06   
    5           NaN            NaN             7.754218e-06   
  4 1           NaN            NaN             2.407270e-06   
    2           NaN            NaN             4.362145e-07   
    3           NaN            NaN             0.000000e+00   
    4           NaN            NaN             2.388054e-07   
    5           NaN            NaN             6.144260e-06   
  5 1           NaN            NaN             1.138530e-06   
    2           NaN            NaN             2.177602e-07   
    3           NaN            NaN             0.000000e+00   
    4           NaN            NaN             1.910842e-07   
    5           NaN            NaN             8.478591e-06   
  6 1           NaN            NaN             0.000000e+00   
    2           NaN            NaN             0.000000e+00   
    3           NaN            NaN             0.000000e+00   
    4           NaN            NaN             0.000000e+00   
    5           NaN            NaN             0.000000e+00   
  7 1           NaN            NaN             0.000000e+00   
    2           NaN            NaN             0.000000e+00   
    3           NaN            NaN             0.000000e+00   
    4           NaN            NaN             0.000000e+00   
    5           NaN            NaN             0.000000e+00   

          primary_cation_flux_vr * dt  cec_cation_flux_vr * dt  \
c j icat                                                         
1 1 1                             0.0                 0.000069   
    2                             0.0                 0.000020   
    3                             0.0                 0.002040   
    4                             0.0                -0.000014   
    5                             0.0                -0.000071   
  2 1                             0.0                 0.000017   
    2                             0.0                 0.000006   
    3                             0.0                 0.000109   
    4                             0.0                 0.000055   
    5                             0.0                -0.000011   
  3 1                             0.0                -0.000035   
    2                             0.0                -0.000007   
    3                             0.0                -0.000073   
    4                             0.0                -0.000014   
    5                             0.0                -0.000033   
  4 1                             0.0                -0.000067   
    2                             0.0                -0.000013   
    3                             0.0                -0.000159   
    4                             0.0                -0.000042   
    5                             0.0                -0.000056   
  5 1                             0.0                -0.000081   
    2                             0.0                -0.000014   
    3                             0.0                -0.000122   
    4                       

In [43]:
collected_lines['Post-reaction cec cation']

cec_cation_vr  meq cec_cation_vr  -cec_cation_flux_vr * dt  \
c j icat                                                               
1 1 1        171.681591           0.591776                 -0.000069   
    2         15.103512           0.085846                 -0.000020   
    3         16.856354           0.050645                 -0.002040   
    4         37.650454           0.066515                  0.000014   
    5          3.791674           0.029122                  0.000071   
  2 1         82.885337           0.285701                 -0.000017   
    2          9.601474           0.054573                 -0.000006   
    3         12.797424           0.038450                 -0.000109   
    4         26.010460           0.045951                 -0.000055   
    5         16.130747           0.123892                  0.000011   
  3 1         66.600572           0.229568                  0.000035   
    2          8.916390           0.050680                  0.000007   
    3         31.012686           0.093177                  0.000073   
    4         13.819799           0.024415                  0.000014   
    5         29.181402           0.224127                  0.000033   
  4 1          5.743038           0.019796                  0.000067   
    2          0.773646           0.004397                  0.000013   
    3          8.384437           0.025191                  0.000159   
    4          1.020451           0.001803                  0.000042   
    5          7.797352           0.059887                  0.000056   
  5 1          8.048913           0.027744                  0.000081   
    2          0.823576           0.004681                  0.000014   
    3          6.138220           0.018442                  0.000122   
    4          2.214657           0.003913                  0.000034   
    5        277.552338           2.131737                  0.000274   
  6 1               NaN                NaN                       NaN   
    2               NaN                NaN                       NaN   
    3               NaN                NaN                       NaN   
    4               NaN                NaN                       NaN   
    5               NaN                NaN                       NaN   
  7 1        321.119725           1.106880                  0.000000   
    2         57.782071           0.328426                  0.000000   
    3        528.402077           1.587578                  0.000000   
    4        184.772867           0.326430                  0.000000   
    5        411.530641           3.160756                  0.000000   

          -cec_cation_flux2_vr * dt  background_cec_vr * dt  
c j icat                                                     
1 1 1                      0.000000                0.001469  
    2                      0.000000                0.000350  
    3                      0.000000                0.013375  
    4                      0.000000                0.000936  
    5                      0.000000                0.000250  
  2 1                      0.000000                0.000651  
    2                      0.000000                0.000144  
    3                      0.000000                0.000090  
    4                      0.000000                0.000558  
    5                      0.000000                0.000252  
  3 1                      0.000000                0.000453  
    2                      0.000000                0.000092  
    3                      0.000000                0.000000  
    4                      0.000000                0.000279  
    5                      0.000000                0.000147  
  4 1                      0.000000                0.000141  
    2                      0.000000                0.000027  
    3                      0.000000                0.000000  
    4                      0.000000                0.000034  
    5                      0.000000        